# 🇮🇳 VoiceGuard India - Kaggle Native Pipeline

Because HuggingFace deleted or moved several of their datasets (causing the 404 errors), we are switching to the **Ultimate Hackathon Fallback**: Native Kaggle Datasets!

These datasets are hosted directly by Kaggle, meaning they require zero HuggingFace tokens, zero 'Agree to Terms' buttons, and zero internet streaming. They load instantly at 5Gbps.

### ⚠️ REQUIRED SETUP (Do this before running!)
1. Look at the right sidebar in Kaggle and click **+ Add Data**.
2. Search for exactly this and click Add: `Hindi Speech TTS Dataset` (by ashutosh0071)
3. Click **+ Add Data** again.
4. Search for exactly this and click Add: `Hindi speech classification` (by vivek468)
5. Once both are attached, run the cells below!

In [ ]:
!pip install librosa scikit-learn numpy tqdm xgboost soundfile

In [ ]:
import os
import librosa
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("\n--- DIAGNOSTIC: Checking for Attached Datasets ---")
authentic_files = []
deepfake_files = []

for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file.endswith(('.wav', '.mp3', '.flac')):
            path = os.path.join(root, file)
            # Route to Authentic or Deepfake based on folder name
            if 'hindi-speech-classification' in path:
                authentic_files.append(path)
            elif 'hindi-speech-tts' in path:
                deepfake_files.append(path)

print(f"✅ Found {len(authentic_files)} Authentic Human Hindi files!")
print(f"✅ Found {len(deepfake_files)} Deepfake (TTS) Hindi files!")

if len(authentic_files) == 0 or len(deepfake_files) == 0:
    raise Exception("❌ ERROR: Datasets are missing! You MUST click '+ Add Data' and add BOTH datasets listed in the instructions!")

# Balance the dataset to prevent bias
max_samples = min(2000, len(authentic_files), len(deepfake_files))
authentic_files = authentic_files[:max_samples]
deepfake_files = deepfake_files[:max_samples]

In [ ]:
def extract_features(audio_path):
    try:
        # Load only 3 seconds to ensure fast processing
        y, sr = librosa.load(audio_path, sr=16000, duration=3.0)
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        return np.hstack((mfccs, centroid, zcr))
    except:
        return None

print("🎙️ Extracting Acoustic Features from Authentic Human Audio...")
with ThreadPoolExecutor() as executor:
    X_real = list(tqdm(executor.map(extract_features, authentic_files), total=len(authentic_files)))
X_real = [x for x in X_real if x is not None]
y_real = [1] * len(X_real)

print("🤖 Extracting Acoustic Features from AI Deepfake Audio...")
with ThreadPoolExecutor() as executor:
    X_fake = list(tqdm(executor.map(extract_features, deepfake_files), total=len(deepfake_files)))
X_fake = [x for x in X_fake if x is not None]
y_fake = [0] * len(X_fake)

X = np.array(X_real + X_fake)
y = np.array(y_real + y_fake)

In [ ]:
print("\n🚀 Training XGBoost Classifier on Native Kaggle Data...")
clf = xgb.XGBClassifier(n_estimators=200, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42)
clf.fit(X, y)

print("\n✅ Training Complete! Accuracy Metrics:")
predictions = clf.predict(X)
print(classification_report(y, predictions, target_names=["Deepfake", "Authentic"]))

with open("voiceguard_rf_model.pkl", "wb") as f:
    pickle.dump(clf, f)
    
print("\n💾 SUCCESS! Model saved as 'voiceguard_rf_model.pkl'")